In [1]:
import os
import ssl
import certifi
import urllib3

# Configure SSL certificates
cert_path = certifi.where()
os.environ['SSL_CERT_FILE'] = cert_path
os.environ['REQUESTS_CA_BUNDLE'] = cert_path
os.environ['AWS_CA_BUNDLE'] = cert_path
os.environ['CURL_CA_BUNDLE'] = cert_path

# Create SSL context with proper certificates
ssl_context = ssl.create_default_context(cafile=cert_path)
ssl._create_default_https_context = lambda: ssl_context

# Disable SSL warnings (optional)
urllib3.disable_warnings(urllib3.exceptions.InsecureRequestWarning)

print(f"✅ SSL certificates configured using: {cert_path}")
print("✅ Environment variables set for AWS, requests, and curl")
print("✅ Ready to make secure HTTPS connections!")

✅ SSL certificates configured using: /Users/manojskr/Documents/Code/GitHub/graphrag-toolkit/.venv/lib/python3.10/site-packages/certifi/cacert.pem
✅ Environment variables set for AWS, requests, and curl
✅ Ready to make secure HTTPS connections!


In [2]:
%reload_ext dotenv
%dotenv ../.env


import os

from graphrag_toolkit.lexical_graph import set_logging_config
from graphrag_toolkit.lexical_graph import LexicalGraphQueryEngine
from graphrag_toolkit.lexical_graph.storage import GraphStoreFactory
from graphrag_toolkit.lexical_graph.storage import VectorStoreFactory

set_logging_config('INFO')

graph_store = GraphStoreFactory.for_graph_store(os.environ['GRAPH_STORE'])
vector_store = VectorStoreFactory.for_vector_store(os.environ['VECTOR_STORE'])

query_engine = LexicalGraphQueryEngine.for_traversal_based_search(
    graph_store, 
    vector_store,
    streaming=True
)

response = query_engine.query("What could cause the Lo level voltage to be elevated to 3V?")

print(f"""{response.print_response_stream()}

retrieve_ms: {int(response.metadata['retrieve_ms'])}
answer_ms  : {int(response.metadata['answer_ms'])}
total_ms   : {int(response.metadata['total_ms'])}
""")

Based on the search results, the elevated Lo level voltage of 3V on output pin 11 could be caused by an electrical issue with the device. The customer reported that the low level of output pin 11 is high, indicating that the voltage is not at the expected 0V level. [Source: data/pdfs/QEM-CCR-2410-00001-VR(FQE).pdf (QEM-CCR-2410-00001-VR(FQE).pdf, 11, 1, 1)]

However, the search results also indicate that the TI electrical testing could not verify the customer-reported issue, and no anomaly was observed during the bench analysis. [Source: data/pdfs/QEM-CCR-2410-00001-VR(FQE).pdf (QEM-CCR-2410-00001-VR(FQE).pdf, 11, 6, 6)]

The search results do not provide a definitive cause for the elevated Lo level voltage. The information suggests that the issue could not be reproduced or confirmed during the investigation, and the customer return has been deemed Trouble Not Identified (TNI). [Source: data/pdfs/QEM-CCR-2410-00001-VR(FQE).pdf (QEM-CCR-2410-00001-VR(FQE).pdf, 11, 6, 6)]None

retrieve_m

In [3]:
for n in response.source_nodes:
    print(n.text)

{
  "source": "data/pdfs/QEM-CCR-2410-00001-VR(FQE).pdf (QEM-CCR-2410-00001-VR(FQE).pdf, 11, 1, 1)",
  "topic": "Device Analysis Information",
  "statements": [
    "The Customer Reported Failure Mode is that the Lo level of output pin 11 is high.",
    "The Date Submitted is 2024-10-02.",
    "The Quantity Submitted is 1.",
    "The Customer Contact is the Quality Assurance Dept., Parts Evaluation Division.",
    "The Device Type is AM26LS32ACPWR.",
    "MITSUBISHI ELECTRIC CORPORATION is the customer.",
    "SHE is the Fab Site.",
    "The Flow Type is Customer Return.",
    "The Measurement value is approximately.",
    "The Technology is unknown."
  ]
}
{
  "source": "data/pdfs/QEM-CCR-2410-00001-VR(FQE).pdf (QEM-CCR-2410-00001-VR(FQE).pdf, 11, 11, 11)",
  "topic": "Semiconductor Manufacturing Terms",
  "statements": [
    "VTP Voltage Threshold P",
    "WLP Wafer Level Package",
    "WLR Wafer Level Reliability",
    "XIVA (LSIM) Laser Signal Injection Microscopy (LSIM) is a curre

In [4]:
from graphrag_toolkit.lexical_graph.retrieval.model import SearchResult

def get_query_params_for_results(response, include_sources=True, include_facts=True, limit=-1):

    statement_ids = []
    source_params = []
    fact_params = []
    
    nodes = response[:limit] if isinstance(response, list) else response.source_nodes[:limit]
    
    for n in nodes:
        
        search_result = SearchResult.model_validate(n.metadata)
        source_id = search_result.source.sourceId
        
        for topic in search_result.topics:
            
            for statement in topic.statements:
                
                statement_id = statement.statementId
                chunk_id = statement.chunkId
                
                statement_ids.append(statement_id)
                if include_sources:
                    source_params.append({'s': source_id, 'c': chunk_id, 'l': statement_id})
                if include_facts:
                    fact_params.append(statement_id)
                    
    
    query_parameters = { 
        'statement_ids': statement_ids,
        'source_params': source_params,
        'fact_params': fact_params
    }
    
    return query_parameters
    
query_parameters = get_query_params_for_results(response, limit=10)

In [5]:
query_parameters

{'statement_ids': ['0d3a4a07d76ee28c6c66ddfe8a8de92f',
  '9a8774e42eef267400c4fa66d295b0d9',
  '349fa9580ea4735fad10542f2684384f',
  'd63945b1e5823325cdb79f6da031039f',
  '77fdfe1b2725216f44f869393d708038',
  '4ed1b704f323c1f3bfaebd4bc0dc69cd',
  'a55364891f15adf22064d12b8e61bb0b',
  'c60eb1657c3966a42815d7865679d8ca',
  'cb0e7c4132a29db514438eb347c21e05',
  '0258a160dc0ce8c53a3daaf955b74120',
  '29f7ea5bbb68bae99ba2bd1ab3cfc8bd',
  'b362cd0a9b3b9240c9470f27e3e1bff0',
  '9e80abc29a2299bddf799d2867da0bb9',
  '765000dbcbeb7a88412b84d3c2fc10fc',
  'fc512069e3dc94816766d8b2ae498317',
  'df3274cbd22f484b9af2e691e91e5ed1',
  '7bfc066b44743ddd59edfeafc0c80fdf',
  'a5169988d6cf5f45a04ce0ce93b3334b',
  'd873e173954425b9cb338ac2cf3b4fae',
  '9080b278b73360fc78a001b2f9a4b76a',
  'bb32fce9c2e65373448f92c4b83a0d89',
  '04316f64760cd4c4f592b031eae37254',
  '036bc8d3c07dcfb57419f54ecec77e1d',
  'a387c5d571f5010b471116387635de69',
  '0fed6c7c168ce735632c62c5d37a55e2',
  '40b369aabc1fd5e1cd4bae7e407d85

In [ ]:
# Test graph-notebook import
import graph_notebook
print(f"Graph Notebook version: {graph_notebook.__version__}")
print("✅ Graph Notebook imported successfully!")

Graph Notebook version: 5.0.1
✅ Graph Notebook imported successfully!


In [ ]:
# Load graph-notebook magic commands
%load_ext graph_notebook.magics
print("✅ Graph Notebook magic commands loaded!")

AttributeError: module 'numpy' has no attribute 'int'.
`np.int` was a deprecated alias for the builtin `int`. To avoid this error in existing code, use `int` by itself. Doing this will not modify any behavior and is safe. When replacing `np.int`, you may wish to use e.g. `np.int64` or `np.int32` to specify the precision. If you wish to review your current use, check the release note link for additional information.
The aliases was originally deprecated in NumPy 1.20; for more details and guidance see the original release note at:
    https://numpy.org/devdocs/release/1.20.0-notes.html#deprecations

In [8]:
display_var = '{"__Source__":"url","__Chunk__":"value","__Topic__":"value","__Statement__":"value","__Fact__":"value"}'


In [9]:
%%oc --query-parameters query_parameters -d $display_var -l 20

UNWIND $source_params AS source_params
MATCH p=(s:`__Source__`)<--(c:`__Chunk__`)<--(t:`__Topic__`)<--(l:`__Statement__`)
WHERE id(s) = source_params.s 
    AND id(c) = source_params.c 
    AND id(l) = source_params.l
RETURN p
UNION
MATCH p=(x:`__Source__`)<--(:`__Chunk__`)<--(:`__Topic__`)<--(l:`__Statement__`)<-[:`__SUPPORTS__`]-(:`__Fact__`)-[:`__NEXT__`*0..1]->(:`__Fact__`)-[:`__SUPPORTS__`]->(ll:`__Statement__`)-->(:`__Topic__`)-->(:`__Chunk__`)-->(y:`__Source__`)
WHERE id(l) IN $fact_params
    AND id(ll) IN $fact_params
    AND x <> y
RETURN p
UNION
MATCH p=(l:`__Statement__`)
WHERE id(l) IN $statement_ids
RETURN p

UsageError: Cell magic `%%oc` not found.
